# Fundamentos de Neo4j y Bases de Datos de Grafos

## Taller Educativo: Introducción

Autor: mCárdenas @2025

Este cuaderno cubre los **fundamentos teóricos** que necesitas antes de empezar a trabajar con Neo4j.

### Objetivos de Aprendizaje

1. **Fundamentos de Grafos**: Qué son y por qué son útiles
2. **Neo4j**: Arquitectura y características
3. **Cypher**: Lenguaje de consulta para grafos
4. **Modelado de Datos**: Cómo diseñar modelos de grafo
5. **Caso de Uso**: Fraude de IVA - por qué es ideal para grafos



## 1. ¿Qué es un Grafo?

### 1.1 Definición

Un **grafo** es una estructura matemática que representa relaciones entre entidades.

**Componentes**:
- **Nodos (Vértices)**: Las entidades u objetos
- **Relaciones (Aristas)**: Las conexiones entre nodos
- **Propiedades**: Atributos de nodos y relaciones

### 1.2 Ejemplo Visual

```
(Empresa A)-[:EMITE_FACTURA {monto: 10000, fecha: '2024-01-15'}]->(Empresa B)
```

- **Nodo**: `(Empresa A)` y `(Empresa B)`
- **Relación**: `[:EMITE_FACTURA]`
- **Propiedades**: `{monto: 10000, fecha: '2024-01-15'}`

### 1.3 Tipos de Grafos

#### Grafo Dirigido
Las relaciones tienen dirección:
```
(A)-[:SIGUE]->(B)  // A sigue a B
```

#### Grafo Etiquetado
Nodos y relaciones tienen tipos:
```
(p:Persona)-[:TRABAJA_EN]->(e:Empresa)
```

#### Property Graph (Neo4j)
Nodos y relaciones pueden tener propiedades:
```
(p:Persona {nombre: 'Juan', edad: 30})
  -[:TRABAJA_EN {desde: 2020, cargo: 'Director'}]->
(e:Empresa {nombre: 'Acme Corp'})
```



## 2. ¿Por qué Grafos para Detección de Fraude?

### 2.1 Ventajas de los Grafos

#### 1. Relaciones de Primera Clase

En bases de datos relacionales:
```sql
-- Difícil: Las relaciones son indirectas
SELECT * FROM empresas e1
JOIN facturas f ON e1.id = f.emisor_id
JOIN empresas e2 ON f.receptor_id = e2.id
WHERE ...
```

En grafos:
```cypher
// Natural: Las relaciones son explícitas
MATCH (e1:Empresa)-[:EMITE_FACTURA]->(e2:Empresa)
RETURN e1, e2
```

#### 2. Performance en Consultas de Relaciones

- **SQL**: JOIN es O(n²) - escanea tablas completas
- **Grafo**: Navegar relación es O(1) - punteros directos

#### 3. Consultas de Caminos

Encontrar caminos en SQL requiere CTEs recursivos complejos.

En Neo4j:
```cypher
// Encontrar caminos de hasta 5 saltos
MATCH path = (a)-[:EMITE_FACTURA*1..5]->(b)
RETURN path
```

#### 4. Detección de Patrones

**Ciclos** (fraude carrusel):
```cypher
MATCH (e)-[:EMITE_FACTURA*3..5]->(e)
RETURN e
```

En SQL esto requeriría múltiples self-joins recursivos.

### 2.2 Casos de Uso Ideales para Grafos

✅ **Detección de fraude**: Redes, patrones, ciclos  
✅ **Recomendaciones**: "Amigos de amigos", similitud  
✅ **Redes sociales**: Conexiones, influencia  
✅ **Supply chain**: Trazabilidad, dependencias  
✅ **Knowledge graphs**: Entidades relacionadas  
✅ **Network management**: Infraestructura, dependencias  



## 3. Neo4j: Base de Datos de Grafos Líder

### 3.1 ¿Qué es Neo4j?

Neo4j es la base de datos de grafos más popular del mundo.

**Características principales**:

- **ACID**: Transacciones completas (Atomicidad, Consistencia, Aislamiento, Durabilidad)
- **Nativo de Grafos**: Diseñado desde cero para grafos, no es una capa sobre SQL
- **Escalable**: Soporta miles de millones de nodos y relaciones
- **Cypher**: Lenguaje declarativo específico para grafos
- **Ecosistema Rico**: APOC (procedimientos), GDS (algoritmos), visualización

### 3.2 Arquitectura de Neo4j

```
┌─────────────────────────────────┐
│   Neo4j Browser / Aplicación   │
└────────────┬────────────────────┘
             │
┌────────────▼────────────────────┐
│     Cypher Query Engine         │
└────────────┬────────────────────┘
             │
┌────────────▼────────────────────┐
│   Graph Storage Engine          │
│   (Index-free adjacency)        │
└─────────────────────────────────┘
```

**Index-free adjacency**: Cada nodo tiene punteros directos a nodos relacionados.

### 3.3 Ediciones de Neo4j

- **Community Edition**: Gratuita, single-node
- **Enterprise Edition**: Clustering, backup avanzado, seguridad
- **AuraDB**: Cloud managed (DBaaS)

### 3.4 Instalación

**Opción 1: Neo4j Desktop** (Recomendado para desarrollo)
- Descarga de https://neo4j.com/download/
- GUI amigable, multi-database

**Opción 2: Docker**
```bash
docker run -d \
  --name neo4j \
  -p 7474:7474 -p 7687:7687 \
  -e NEO4J_AUTH=neo4j/password \
  neo4j:latest
```

**Puertos**:
- `7474`: HTTP (Neo4j Browser)
- `7687`: Bolt (protocolo binario para drivers)



## 4. Cypher: El Lenguaje de Neo4j

### 4.1 ¿Qué es Cypher?

**Cypher** es un lenguaje declarativo para consultar grafos, similar a SQL pero específico para propiedades de grafo.

**Filosofía**: Consultas que se leen como diagramas ASCII.

### 4.2 Sintaxis Básica

#### Crear Nodos

```cypher
CREATE (e:Empresa {nombre: "Acme Corp", nif: "A12345678"})
```

- `CREATE`: Crear
- `(e:Empresa ...)`: Variable `e`, etiqueta `Empresa`
- `{...}`: Propiedades

#### Buscar Nodos

```cypher
MATCH (e:Empresa)
WHERE e.nif = "A12345678"
RETURN e.nombre
```

O simplificado:
```cypher
MATCH (e:Empresa {nif: "A12345678"})
RETURN e.nombre
```

#### Crear Relaciones

```cypher
MATCH (a:Empresa {nombre: "Acme Corp"})
MATCH (b:Empresa {nombre: "Beta Inc"})
CREATE (a)-[:EMITE_FACTURA {monto: 5000, fecha: date('2024-01-15')}]->(b)
```

#### Consultar con Relaciones

```cypher
// Empresas que facturan a Beta Inc
MATCH (emisor:Empresa)-[:EMITE_FACTURA]->(receptor:Empresa {nombre: "Beta Inc"})
RETURN emisor.nombre
```

```cypher
// Con filtros en relación
MATCH (e1:Empresa)-[f:EMITE_FACTURA]->(e2:Empresa)
WHERE f.monto > 10000
RETURN e1.nombre, e2.nombre, f.monto
ORDER BY f.monto DESC
```

### 4.3 Patrones Avanzados

#### Caminos Variables

```cypher
// Caminos de 1 a 3 saltos
MATCH path = (e1:Empresa)-[:EMITE_FACTURA*1..3]->(e2:Empresa)
RETURN path
```

#### Ciclos

```cypher
// Detectar empresas en ciclos
MATCH (e:Empresa)-[:EMITE_FACTURA*3..5]->(e)
RETURN e.nombre
```

#### Agregaciones

```cypher
// Volumen total por empresa
MATCH (e:Empresa)-[f:EMITE_FACTURA]->()
RETURN e.nombre, sum(f.monto) AS total
ORDER BY total DESC
```

### 4.4 Comparación SQL vs Cypher

| Operación | SQL | Cypher |
|-----------|-----|--------|
| Buscar | SELECT | MATCH + RETURN |
| Filtrar | WHERE | WHERE |
| Crear | INSERT | CREATE |
| Actualizar | UPDATE | SET |
| Eliminar | DELETE | DELETE / DETACH DELETE |
| Join | JOIN ON | Pattern matching |

**Ejemplo equivalente**:

SQL:
```sql
SELECT e1.nombre, e2.nombre, f.monto
FROM empresas e1
JOIN facturas f ON e1.id = f.emisor_id
JOIN empresas e2 ON f.receptor_id = e2.id
WHERE f.monto > 10000;
```

Cypher:
```cypher
MATCH (e1:Empresa)-[f:EMITE_FACTURA]->(e2:Empresa)
WHERE f.monto > 10000
RETURN e1.nombre, e2.nombre, f.monto
```



## 5. Modelado de Datos en Grafos

### 5.1 Principios de Modelado

#### 1. Nodos = Entidades
Las "cosas" importantes del dominio.

#### 2. Relaciones = Verbos
Cómo se conectan las entidades.

#### 3. Propiedades = Atributos
Información sobre nodos y relaciones.

### 5.2 Ejemplo: Fraude de IVA

#### Entidades (Nodos)

```
(:Empresa)
(:Directivo)
(:CuentaBancaria)
```

#### Relaciones

```
(Directivo)-[:ADMINISTRA]->(Empresa)
(Empresa)-[:EMITE_FACTURA]->(Empresa)
(Empresa)-[:TIENE_CUENTA]->(CuentaBancaria)
```

#### Modelo Completo

```
     (Directivo)
          |
     [:ADMINISTRA]
          |
          ↓
     (Empresa)----[:TIENE_CUENTA]---->(CuentaBancaria)
          |
  [:EMITE_FACTURA]
          |
          ↓
     (Empresa)
```

### 5.3 Propiedades de Nodos

#### Empresa
```cypher
{
  nif: String,
  nombre: String,
  pais: String,
  fecha_constitucion: Date,
  capital_social: Float,
  empleados: Integer
}
```

#### Directivo
```cypher
{
  dni: String,
  nombre: String,
  fecha_nacimiento: Date
}
```

### 5.4 Propiedades de Relaciones

```cypher
-[:EMITE_FACTURA {
  numero_factura: String,
  fecha: Date,
  monto_base: Float,
  iva: Float,
  monto_total: Float,
  concepto: String
}]->
```

**Por qué en la relación**: La factura ES la relación entre empresas.

### 5.5 Relacional vs Grafo

#### Modelo Relacional
```
Tabla: empresas (id, nombre, nif, ...)
Tabla: facturas (id, emisor_id, receptor_id, monto, ...)
Tabla: directivos (id, nombre, dni, ...)
Tabla: administraciones (directivo_id, empresa_id, cargo, ...)
```

**Problema**: Las relaciones son indirectas (foreign keys).

#### Modelo de Grafo
```
(Empresa)-[:EMITE_FACTURA]->(Empresa)
(Directivo)-[:ADMINISTRA]->(Empresa)
```

**Ventaja**: Las relaciones son explícitas y navigables.



## 6. Caso de Uso: Fraude de IVA

### 6.1 ¿Qué es el Fraude de IVA?

Evasión del Impuesto sobre el Valor Añadido mediante prácticas fraudulentas.

**Impacto**: Miles de millones de euros anuales en la UE.

### 6.2 Tipos de Fraude

#### Fraude Carrusel (Intracomunitario)

El más sofisticado:

1. **Empresa A** (España) vende a **Empresa B** (Francia) - Sin IVA (intracomunitario)
2. **Empresa B** vende a **Empresa C** (Francia) - Con IVA, pero B no lo paga
3. **Empresa C** vende a **Empresa D** (España) - Sin IVA
4. **Ciclo se repite**: A → B → C → D → A

**Empresa B** es "missing trader" - desaparece sin pagar IVA.

#### Empresas Fantasma

Características:
- Creadas recientemente
- Capital social mínimo (3.000€)
- Sin empleados o muy pocos
- Directivos compartidos
- Alto volumen en poco tiempo
- Desaparecen rápidamente

#### Facturación Falsa

- Facturas por servicios inexistentes
- Red de empresas que se facturan mutuamente
- Para deducir IVA falso

### 6.3 Por qué Grafos son Ideales

Los patrones de fraude son **relacionales**:

✅ **Ciclos**: A→B→C→A (fraude carrusel)  
✅ **Redes**: Directivos con múltiples empresas  
✅ **Comunidades**: Grupos cerrados de empresas  
✅ **Flujos**: Dinero entre países  
✅ **Caminos**: Trazabilidad de transacciones  

### 6.4 Consultas que Serían Difíciles en SQL

#### 1. Ciclos de 3-5 empresas
```cypher
MATCH (e)-[:EMITE_FACTURA*3..5]->(e)
RETURN e
```

En SQL: CTEs recursivos complejos.

#### 2. Directivos con empresas que comercian
```cypher
MATCH (d:Directivo)-[:ADMINISTRA]->(e1:Empresa)
MATCH (d)-[:ADMINISTRA]->(e2:Empresa)
MATCH (e1)-[:EMITE_FACTURA]-(e2)
WHERE e1 <> e2
RETURN d, e1, e2
```

En SQL: Múltiples self-joins.

#### 3. Comunidades cerradas
```cypher
MATCH (e:Empresa)-[:EMITE_FACTURA*]-(otras:Empresa)
// Detectar grupos que solo comercian entre sí
```

En SQL: Extremadamente complejo.



## 7. Preparación para el Taller

### 7.1 Lo que Necesitas

✅ **Neo4j instalado y corriendo**
- Desktop, Docker, o Aura
- Puertos 7474 y 7687 accesibles

✅ **Plugins instalados**
- APOC (Awesome Procedures on Cypher)
- GDS (Graph Data Science)

✅ **Python y bibliotecas**
```bash
pip install -r requirements.txt
```

✅ **Datos generados**
```bash
python generar_datos_fraude.py
```

### 7.2 Estructura del Taller

**Notebook 00** (este): Fundamentos teóricos  
**Notebook 01**: Detección de fraude con Cypher  
**Notebook 02**: Procedimientos avanzados con APOC  
**Notebook 03**: Algoritmos de graph data science  

### 7.3 Verificar Instalación

In [ ]:
# Verificar conexión a Neo4j
from neo4j import GraphDatabase

NEO4J_URI = "bolt://localhost:7687"
NEO4J_USER = "neo4j"
NEO4J_PASSWORD = "abc123456"  # CAMBIAR

try:
    driver = GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USER, NEO4J_PASSWORD))
    with driver.session() as session:
        result = session.run("RETURN 'OK' AS status")
        print(f"✅ Neo4j: {result.single()['status']}")
    driver.close()
except Exception as e:
    print(f"❌ Error: {e}")

In [ ]:
# Verificar APOC
from neo4j import GraphDatabase

driver = GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USER, NEO4J_PASSWORD))
try:
    with driver.session() as session:
        result = session.run("RETURN apoc.version() AS version")
        print(f"✅ APOC: {result.single()['version']}")
except Exception as e:
    print(f"❌ APOC no instalado: {e}")
finally:
    driver.close()

In [ ]:
# Verificar GDS
driver = GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USER, NEO4J_PASSWORD))
try:
    with driver.session() as session:
        result = session.run("RETURN gds.version() AS version")
        print(f"✅ GDS: {result.single()['version']}")
except Exception as e:
    print(f"❌ GDS no instalado: {e}")
finally:
    driver.close()

## 8. Resumen y Próximos Pasos

### Lo que Aprendiste

✅ Qué son los grafos y cuándo usarlos  
✅ Arquitectura y características de Neo4j  
✅ Sintaxis básica de Cypher  
✅ Cómo modelar datos en grafos  
✅ Por qué los grafos son ideales para fraude  

### Conceptos Clave

- **Property Graph**: Nodos y relaciones con propiedades
- **Index-free Adjacency**: Performance O(1) en navegación
- **Cypher**: Lenguaje declarativo, fácil de leer
- **Pattern Matching**: El núcleo de las consultas

### Próximos Pasos

**Continúa con el Notebook 01**:
- Cargar datos reales en Neo4j
- Ejecutar consultas Cypher
- Detectar los 6 patrones de fraude
- Ver la potencia de Neo4j en acción


## Recursos Adicionales

### Documentación
- [Neo4j Documentation](https://neo4j.com/docs/)
- [Cypher Manual](https://neo4j.com/docs/cypher-manual/current/)
- [Cypher Refcard](https://neo4j.com/docs/cypher-refcard/current/)

### Tutoriales
- [GraphAcademy](https://graphacademy.neo4j.com/) - Cursos gratuitos
- [Neo4j Sandbox](https://sandbox.neo4j.com/) - Entorno de prueba

### Comunidad
- [Neo4j Community Forum](https://community.neo4j.com/)
- [Stack Overflow - neo4j](https://stackoverflow.com/questions/tagged/neo4j)



**¡Ahora estás listo para empezar con Neo4j!** 🚀